In [1]:
import torch
import random
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

# 1. 데이터 생성 함수
def generate_good_code(i):
    templates = [
        f"def add_{i}(a, b):\n    return a + b",
        f"def check_{i}(x):\n    return x > 0",
        f"def greet_{i}(name):\n    return f'Hello {{name}}'",
        f"def square_{i}(x):\n    return x ** 2",
        f"def is_even_{i}(n):\n    return n % 2 == 0",
    ]
    return random.choice(templates)

def generate_bad_code(i):
    templates = [
        f"def f(a,b,c):\n    for i in range(a):\n        for j in range(b):\n            print(i,j)",
        f"def x(a,b,c,d,e,f):\n    return a+b+c+d+e+f",
        f"def bad_{i}(x):\n    a=1\n    b=2\n    c=3\n    d=4\n    e=5\n    return x+a+b+c+d+e",
        f"def g(x):\n    for i in range(x):\n        for j in range(x):\n            for k in range(x):\n                print(i,j,k)",
    ]
    return random.choice(templates)

# 2. 데이터 준비
good = [generate_good_code(i) for i in range(25)]
bad = [generate_bad_code(i) for i in range(25)]
codes = good + bad
labels = [round(random.uniform(0.7, 0.95), 2) for _ in range(25)] + \
         [round(random.uniform(0.05, 0.35), 2) for _ in range(25)]

# 3. 토크나이저
all_chars = sorted(set(''.join(codes)))
char2idx = {ch: idx+1 for idx, ch in enumerate(all_chars)}
idx2char = {idx: ch for ch, idx in char2idx.items()}
vocab_size = len(all_chars) + 1

def encode(code, max_len=100):
    encoded = [char2idx[c] for c in code if c in char2idx]
    encoded = encoded[:max_len] + [0] * (max_len - len(encoded))
    return encoded

# 4. Dataset, DataLoader
X = torch.tensor([encode(code) for code in codes], dtype=torch.long)
y = torch.tensor(labels, dtype=torch.float32)

class CodeDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = CodeDataset(X, y)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# 5. 모델
class CodeQualityLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        out = self.fc(hidden.squeeze(0))
        return out.squeeze(1)

model = CodeQualityLSTM(vocab_size=vocab_size, embed_dim=32, hidden_dim=64)

# 6. 학습
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(500):
    for x_batch, y_batch in dataloader:
        pred = model(x_batch)
        loss = loss_fn(pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if epoch % 50 == 0:
        print(f"epoch {epoch} | loss: {loss.item():.4f}")

print("학습 완료!")

epoch 0 | loss: 0.0385
epoch 50 | loss: 0.0681
epoch 100 | loss: 0.0423
epoch 150 | loss: 0.0237
epoch 200 | loss: 0.0090
epoch 250 | loss: 0.0035
epoch 300 | loss: 0.0076
epoch 350 | loss: 0.0116
epoch 400 | loss: 0.0037
epoch 450 | loss: 0.0042
학습 완료!


In [2]:
model.eval()

with torch.no_grad():
    correct = 0
    for i, code in enumerate(codes):
        x = torch.tensor([encode(code)], dtype=torch.long)
        pred = model(x).item()
        actual = labels[i]
        ok = '✅' if abs(pred - actual) < 0.2 else '❌'
        if ok == '✅':
            correct += 1
        print(f"실제: {actual:.2f}  예측: {pred:.2f}  {ok}")
    
    print(f"\n정확도: {correct}/{len(codes)} ({correct/len(codes)*100:.1f}%)")

실제: 0.87  예측: 0.82  ✅
실제: 0.73  예측: 0.82  ✅
실제: 0.74  예측: 0.82  ✅
실제: 0.90  예측: 0.82  ✅
실제: 0.74  예측: 0.82  ✅
실제: 0.84  예측: 0.82  ✅
실제: 0.78  예측: 0.82  ✅
실제: 0.88  예측: 0.82  ✅
실제: 0.94  예측: 0.82  ✅
실제: 0.84  예측: 0.82  ✅
실제: 0.93  예측: 0.82  ✅
실제: 0.80  예측: 0.82  ✅
실제: 0.87  예측: 0.82  ✅
실제: 0.89  예측: 0.82  ✅
실제: 0.73  예측: 0.82  ✅
실제: 0.83  예측: 0.82  ✅
실제: 0.93  예측: 0.82  ✅
실제: 0.90  예측: 0.82  ✅
실제: 0.86  예측: 0.82  ✅
실제: 0.77  예측: 0.82  ✅
실제: 0.75  예측: 0.82  ✅
실제: 0.79  예측: 0.82  ✅
실제: 0.72  예측: 0.82  ✅
실제: 0.82  예측: 0.82  ✅
실제: 0.72  예측: 0.82  ✅
실제: 0.35  예측: 0.24  ✅
실제: 0.28  예측: 0.24  ✅
실제: 0.26  예측: 0.23  ✅
실제: 0.23  예측: 0.24  ✅
실제: 0.17  예측: 0.26  ✅
실제: 0.09  예측: 0.23  ✅
실제: 0.32  예측: 0.29  ✅
실제: 0.29  예측: 0.26  ✅
실제: 0.16  예측: 0.23  ✅
실제: 0.17  예측: 0.24  ✅
실제: 0.20  예측: 0.23  ✅
실제: 0.20  예측: 0.24  ✅
실제: 0.21  예측: 0.26  ✅
실제: 0.32  예측: 0.26  ✅
실제: 0.17  예측: 0.24  ✅
실제: 0.31  예측: 0.23  ✅
실제: 0.21  예측: 0.29  ✅
실제: 0.31  예측: 0.29  ✅
실제: 0.34  예측: 0.23  ✅
실제: 0.32  예측: 0.24  ✅
실제: 0.17  